# 쓰레기 이미지 분류 — 예측 데모

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehtel-crypto/KmubigdataTRASH/blob/main/predict_demo.ipynb)

학습이 끝난 최종 모델(**ResNet18 finetune, 테스트 정확도 0.9695**)을 내려받아, 새 사진이 12개 클래스 중 무엇인지 예측합니다.
**학습 과정은 없습니다.** 학습 전체 과정과 실험은 `garbage_classification.ipynb`에 있습니다.

- GPU가 없어도 됩니다 (CPU에서 사진 한 장에 1초 안팎)
- Kaggle 데이터 다운로드, Google Drive 연결이 필요 없습니다
- 처음 실행부터 결과까지 약 1~2분

**사용법**: 위에서부터 셀을 차례로 실행 → 셀 3에서 "파일 선택" 버튼으로 사진 업로드

In [ ]:
# 셀 1. 준비 — 라이브러리, 학습된 모델 내려받기
import os, io
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# 학습에 쓴 클래스 이름 (ImageFolder가 폴더 이름을 알파벳 순으로 정한 순서 그대로)
class_names = ["battery", "biological", "brown-glass", "cardboard", "clothes", "green-glass",
               "metal", "paper", "plastic", "shoes", "trash", "white-glass"]
NUM_CLASSES = len(class_names)

# 학습된 가중치 — GitHub Releases에 올려 둔 파일 (약 45MB)
MODEL_URL  = "https://github.com/Ehtel-crypto/KmubigdataTRASH/releases/download/v1.0/best_model.pt"
MODEL_PATH = "best_model.pt"
if not os.path.exists(MODEL_PATH):
    print("모델 내려받는 중...")
    torch.hub.download_url_to_file(MODEL_URL, MODEL_PATH)
print("모델 파일:", MODEL_PATH, f"({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB)")

In [ ]:
# 셀 2. 모델 만들기 → 학습된 가중치 덮어씌우기 (10주차: ResNet18의 fc를 12클래스로 교체)
model = models.resnet18(weights=None)            # 구조만 만듭니다. 가중치는 아래에서 우리 것으로 채움
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# 학습 때 검증·테스트에 쓴 것과 똑같은 변환
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


def predict(img, top_k=3, show=True):
    """img: 파일 경로 또는 PIL 이미지. 상위 top_k개 (클래스, 확률)을 돌려줍니다."""
    if isinstance(img, str):
        img = Image.open(img)
    img = img.convert("RGB")                        # png 투명 배경·흑백 사진도 3채널로 통일

    x = eval_tf(img).unsqueeze(0).to(device)        # (3, 224, 224) → (1, 3, 224, 224)
    with torch.no_grad():
        prob = torch.softmax(model(x), dim=1)[0].cpu()   # 점수 → 확률 (합이 1)

    top_p, top_i = prob.topk(top_k)
    result = [(class_names[i], round(p.item(), 4)) for p, i in zip(top_p, top_i)]

    if show:
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.5), gridspec_kw={"width_ratios": [1, 1.2]})
        ax[0].imshow(img); ax[0].axis("off")
        ax[0].set_title(f"pred: {result[0][0]} ({result[0][1]:.1%})")
        ax[1].barh([r[0] for r in result][::-1], [r[1] for r in result][::-1])
        ax[1].set_xlim(0, 1); ax[1].set_xlabel("probability"); ax[1].set_title(f"top {top_k}")
        ax[1].grid(axis="x", alpha=.3)
        plt.tight_layout(); plt.show()
    return result

print("준비 완료 — 셀 3에서 사진을 올리세요")

In [ ]:
# 셀 3. 사진 업로드해서 예측 ("파일 선택" 버튼이 나오면 사진을 고르세요. 여러 장 가능)
from google.colab import files

uploaded = files.upload()
for fname in uploaded:
    print(f"\n▶ {fname}")
    res = predict(fname)
    for rank, (c, p) in enumerate(res, 1):
        print(f"  {rank}위  {c:12s} {p:.1%}")

In [ ]:
# 셀 4 (선택). 인터넷 이미지 주소로 바로 예측
# 구글 이미지에서 사진 클릭 → 크게 뜬 사진에서 오른쪽 버튼 → "이미지 주소 복사" → 아래 따옴표 안에 붙여넣기
# 주소가 data: 로 시작하거나 에러가 나면, 사진을 저장해서 셀 3으로 올려 주세요
import requests

url = "여기에 이미지 주소를 붙여넣으세요"

resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
res = predict(Image.open(io.BytesIO(resp.content)))
for rank, (c, p) in enumerate(res, 1):
    print(f"  {rank}위  {c:12s} {p:.1%}")

## 결과를 읽을 때 참고할 점
- **12개 클래스만 압니다.** 쓰레기가 아닌 사진을 넣어도 12개 중 하나로 답합니다. 이때는 1위 확률이 낮게 나오는 경우가 많습니다.
- **1위 확률이 50% 아래면** 모델이 확신하지 못한 결과로 보는 것이 좋습니다.
- **사진 한 장에 답은 하나입니다.** 여러 종류의 물건이 함께 찍혀 있으면 판별할 물건만 잘라서 넣어 주세요.
- 테스트 정확도 0.97은 Kaggle 데이터셋과 비슷한 사진(물건 하나가 크게 찍힌 사진) 기준입니다. 투명 플라스틱 컵은 white-glass로 헷갈리는 경향이 있습니다.